# Introduction

In this use case, we will embark on a journey to explore the capabilities of **SAP HANA Cloud vector engine**, **SAP Generative AI Hub** and the **Langchain** Framework. The goal is to equip you with the knowledge and skills to handle unstructured and semi-structured data and build efficient applications.

Embed structured and semi-structured data using **Large Language Models** (LLMs) from **SAP Generative AI Hub**. Once the data is embedded, it will be stored in **SAP HANA Cloud** helping to store and query vector embeddings seamlessly. 


## About the data set

The data set is a product catalog of IT accessory products. Here are the main attributes and their descriptions based on the sample data:

|Field          |Description            |
----------------|-----------------------
|**PRODUCT_ID**| A unique identifier for each product.|
|**PRODUCT_NAME**| The name of the product, which typically includes the brand and the model.|
|**CATEGORY**| The general category of the product, which is "IT Accessories" for all entries sampled.|
|**DESCRIPTION**| A detailed description of the product, highlighting key features and specifications.|
|**UNIT_PRICE**| The price of the product in Euros.|
|**UNIT_MEASURE**| The unit of measure for the product, typically "Each" indicating pricing per item.|
|**SUPPLIER_ID**| A unique identifier for the supplier of the product.|
|**SUPPLIER_NAME**| The name of the supplier.|
|**LEAD_TIME_DAYS**| The number of days it takes from order to delivery.|
|**MIN_ORDER**| The minimum order quantity required.|
|**CURRENCY**| The currency of the transaction, which is "EURO" for all entries.|
|**SUPPLIER_COUNTRY**| The country where the supplier is located, which is "Germany" for all sampled entries.|
|**SUPPLIER_ADDRESS**| The physical address of the supplier.|
|**AVAILABILITY_DAYS**| The number of days the product is available for delivery.|
|**SUPPLIER_CITY**| The city where the supplier is located.|
|**STOCK_QUANTITY**| The quantity of the product currently in stock.|
|**MANUFACTURER**| The company that manufactured the product.|
|**CITY_LAT**| Geographical coordinates of the city (latitude)|
|**CITY_LONG**| Geographical coordinates of the city (longitude).|
|**RATING:**| A rating for the product, which are on a scale from 1 to 5.|

</br>

> This dataset is structured to support various business operations such as inventory management, order processing, and logistics planning, providing a comprehensive view of product offerings, supplier details, and stock levels. Each entry is highly detailed, suggesting the dataset could be used for analytical purposes, such as optimizing supply chain operations or analyzing sales and marketing strategies.

</br>

## Retrieval Augmented Generation in generative AI Hub using SAP HANA vector search

### Hands-on Retrieval Augmented Generation (RAG) workflow 

The Retrieval Augmented Generation use case process consists of steps to be completed as seen in the graphic below. 

<br>


> ![title](./images/vector_flow_1.png)

<br> 



1. Documents to be included in vector analysis are fed into the model.

2. The contents of the files are split into smaller chunks.

3. Embedding functions are used to create embeddings from the file/document chunks.

4. The embeddings are then stored as vectors in the SAP HANA Cloud Database.

5. When a query or prompt is submitted, the query itself is then embedded into vector form.

6. The query vector is compared to the values stored as vectors in SAP HANA Cloud via a similarity/semantic search.

7. The most appropriate results are forwarded, along with the original query, to a large language model such as Chat GPT.

8. The LLM uses the results of the HANA vector search to augment its own searching capabilities, and the final answer is returned to the user.

### Setup and configuration

- Install required Python modules

### Implementing RAG Embeddings

- Prepare the documentation for the product catalog in CSV format with each row representing a product.

- Connect to the HANA vector storage instance and create a table to store the documentation data.

- Populate the table with data and create a REAL_VECTOR column to store embeddings.

- Use the Generative AI Hub SDK to define a function to generate embeddings for prompts and perform similarity search using the embeddings.

### Enhancing Query Responses

- Define a prompt template to provide context to queries.

- Modify the function to query the LLM (Large Language Model) based on the prompt template.

- Test the model's response using specific queries related to the node library, ensuring it provides contextually relevant responses based on embeddings.

> Retrieval augmented generation optimizes the output of large language models by applying more context to prompts.

</br>

> **Migration note:** This notebook has been migrated to use `genai-proxy-secure` via the `sap-ai-sdk-gen` (`GenAIHubProxyClient`) client. Replace the two `REPLACE_ME` values in the *Proxy configuration* cell with your proxy URL and API key. No AI Core environment variables or service bindings are required.

## Setup and configuration

The following Python modules are to be installed during this hands-on introduction. 

#### **hdbcli**

The Python Database API Specification v2.0 (PEP 249) defines a set of methods that provides a consistent database interface independent of the actual database being used. The Python extension module for SAP HANA implements PEP 249. Once you install the module, you can access and change the information in SAP HANA databases from Python.


#### **sap-ai-sdk-gen**

With this SAP python SDK you can leverage the power of generative Models like chatGPT available in SAP's generative AI Hub.

#### **Folium**

Folium builds on the data wrangling strengths of the Python ecosystem and the mapping strengths of the Leaflet.js library. Manipulate your data in Python, then visualize it in a Leaflet map via folium.


<br>

> **Note:** Jupyter Notebook kernel restart required after package installation.


</br>

#### Install Python packages

Run the following package installations. **pip** is the package installer for Python. You can use pip to install packages from the Python Package Index and other indexes.

In [ ]:
%pip install hdbcli 
%pip install "sap-ai-sdk-gen[all]" 
%pip install folium 
%pip install ipywidgets
%pip install langchain-community
%pip install langchain-hana
%pip install langchain-aws
%pip install langchain-google-genai
%pip install aiobotocore
%pip install python-dotenv

# kernel restart required!!!

In [ ]:
%pip show sap-ai-sdk-gen

#### Restart Python kernel

The Python kernel needs to be restarted before continuing. 

> ![title](./images/config_001.png)

</br>

> **Note** This will take a couple of minutes.

### Proxy configuration

Credentials are loaded automatically from a `.env` file in the project root using `python-dotenv`. Ensure the file contains the following variables:

| Variable | Description |
|---|---|
| `AICORE_BASE_URL` | Proxy base URL **without** `/v1` suffix |
| `AICORE_AUTH_URL` | OAuth token endpoint |
| `AICORE_CLIENT_ID` | OAuth client ID |
| `AICORE_CLIENT_SECRET` | OAuth client secret (used as the API key) |
| `AICORE_RESOURCE_GROUP` | Resource group (e.g. `default`) |

> **Note:** `python-dotenv` must be installed (`pip install python-dotenv`). Make sure `.env` is listed in `.gitignore`.

In [ ]:
import os
from dotenv import load_dotenv
from gen_ai_hub.proxy.gen_ai_hub_proxy.client import GenAIHubProxyClient

load_dotenv()

PROXY_BASE_URL   = os.environ["AICORE_BASE_URL"]
PROXY_API_KEY    = os.environ["AICORE_CLIENT_SECRET"]
PROXY_CLIENT_ID  = os.environ["AICORE_CLIENT_ID"]
PROXY_AUTH_URL   = os.environ["AICORE_AUTH_URL"]
PROXY_RG         = os.environ["AICORE_RESOURCE_GROUP"]

CHAT_MODEL       = "gpt-4o"
EMBEDDING_MODEL  = "text-embedding-3-large"

proxy_client = GenAIHubProxyClient(
    base_url=PROXY_BASE_URL,
    auth_url=PROXY_AUTH_URL,
    client_id=PROXY_CLIENT_ID,
    client_secret=PROXY_API_KEY,
    resource_group=PROXY_RG,
)

# Verify connectivity — lists deployments visible to this key
for d in proxy_client.get_deployments():
    print(d.deployment_id, d.config_name, d.model_name)

#### Configure SAP Generative AI Hub credentials

Credentials are now supplied via `GenAIHubProxyClient` above — no `AICORE_*` environment variables are required. The proxy handles authentication using the `client_secret` (your API key) and routes all model calls through `genai-proxy-secure`.

</br>

> Before continuing, please ensure that the Python kernel was restarted!!

#### Test SAP Generative AI Hub configuration

Run the test below to verify connectivity by making a call to the `text-embedding-3-large` model via `genai-proxy-secure`.

In [ ]:
# Test embeddings via init_embedding_model
from gen_ai_hub.proxy.langchain.init_models import init_embedding_model

embeddings = init_embedding_model(EMBEDDING_MODEL, proxy_client=proxy_client)
test_vec = embeddings.embed_query("SAP Generative AI Hub is awesome!")
print(f"Embedding dimension: {len(test_vec)}")

### Initialize and test the embeddings model
Embeddings are vector representations of text data that incorporate the semantic meaning of the text. The cell below initializes the embeddings object using the **text-embedding-3-large** model via the proxy and runs a quick dimension check to confirm connectivity.

> An deployment for **text-embedding-3-large** in SAP Generative AI foundation-models have been created.

### Initialize the LLM model
LLM is initialized as an instance of ChatOpenAI with a model named **gpt-4o**. This is used for generating responses or interacting in a chat-like environment.

In [ ]:
# Set llm via proxy_client
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI

llm = ChatOpenAI(proxy_model_name=CHAT_MODEL, proxy_client=proxy_client)

### Ask LLM without context

After completing the configuration we are ready to ask the first question directly to LLM (gpt-4o) without any business product context to find us products with a rating of 4 and more. The response is arbitrary and does not relate to our product data. 

</br>

> **Note** We can solve this problem by following the next steps in implementing RAG Embeddings.


In [ ]:
from IPython.display import Markdown

question = "Find 5 products with a rating of 4 and more."
llm.temperature=0.7
response = llm.invoke(question)
display(Markdown(response.content))

## Implementing RAG Embeddings

Now that all SAP Generative AI Hub configuration steps have been completed, let's continue to process the product catalog data file.

### Prepare the documentation for the product catalog in CSV format with each row representing a product

This code snippet demonstrates how to load and process text data from a CSV file using the `CSVLoader` from the `langchain.document_loaders.csv_loader` module.

This process is useful for handling large text data, making it more manageable or suitable for further processing, analysis, or input into machine learning models, especially when dealing with limitations on input size.


In [ ]:
# Process CSV data file

from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path="data/new_product.csv",
    csv_args={
        "delimiter": ";",
        "quotechar": '"',
        "fieldnames": ["PRODUCT_ID","PRODUCT_NAME","CATEGORY","DESCRIPTION","UNIT_PRICE","UNIT_MEASURE","SUPPLIER_ID","SUPPLIER_NAME","LEAD_TIME_DAYS","MIN_ORDER","CURRENCY","SUPPLIER_COUNTRY","SUPPLIER_ADDRESS","AVAILABILITY_DAYS","SUPPLIER_CITY","STOCK_QUANTITY","MANUFACTURER","CITY_LAT","CITY_LONG", "RATING"],
    },
)

# Process data

text_documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
text_chunks = text_splitter.split_documents(text_documents)
print(f"Number of document chunks: {len(text_chunks)}")

for chunks in text_chunks:
    print(chunks.metadata)
    print(chunks.page_content)

### SAP HANA Cloud vector engine

Storing vector embeddings within the same database is a strategic move that aligns seamlessly with SAP's commitment to providing a unified platform. This integration eliminates the hurdles posed by data silos, offering a holistic approach to data management. In SAP HANA Cloud, the storage of vector embeddings is seamlessly integrated into the platform's existing structure, allowing users to store them in a designated table. Developers can perform SQL-like queries effortlessly. 

This means you can execute joins, apply filters, and perform selects by combining vector embeddings with various data types, including transactional, spatial, graph, and JSON data, all within the same SQL environment. The Vector Engine ensures a user-friendly experience, eliminating the need for extensive learning or the adoption of new querying methodologies. Essentially, working with vector embeddings in SAP HANA Cloud is as straightforward as crafting queries in a standard SQL database, offering familiarity and ease of use for developers.

### Connect to the HANA vector storage instance and create a table to store the documentation data

The provided Python script imports database connection modules and initiates a connection to a SAP HANA Cloud instance using the `dbapi` module. The user is prompted to enter their username and password, which are then used to establish a secure connection to the SAP HANA Cloud database. 

The `langchain_community.vectorstores.hanavector` library, specifically the `HanaDB` class, from the LangChain community, is designed to interact with vector data stored in SAP HANA Cloud database, and enables developers to utilize SAP HANA Cloud's advanced capabilities for managing and querying vector data, in the context of AI and machine learning applications.

</br>

> **Note** Use your username and password supplied to logon to the SAP HANA Cloud database. Find the **host_address** in the lesson content.

In [ ]:
# HC Vector Engine

from hdbcli import dbapi
try:
    from langchain_hana import HanaDB
except ImportError:
    from langchain_community.vectorstores.hanavector import HanaDB

# host_address = input("Enter HANA Cloud Hostname")
# hdb_user = input("Enter Username")
# hdb_password = input("Enter Password :")
host_address = "ENTER_HOSTNAME_HERE"
hdb_user = "ENTER_USERNAME_HERE"
hdb_password = "ENTER_PASSWORD_HERE"

connection = dbapi.connect(
    host_address,
    port="443",
    user=hdb_user,
    password=hdb_password,
    autocommit=True,
    sslValidateCertificate=False,
)

### Populate the table with data and creates a REAL_VECTOR column to store embeddings

Create a LangChain VectorStore interface for the HANA database and specify the table (collection) to use for accessing the vector embeddings. Embeddings are vector representations of text data that incorporate the semantic meaning of the text.

In [ ]:
#Create a LangChain VectorStore interface for the HANA database and specify the table (collection) to use for accessing the vector embeddings
db = HanaDB(
    embedding=embeddings, connection=connection, table_name="CATALOG_UPDATED_DEV_1_"+ hdb_user
)

In [ ]:
# Delete already existing documents from the table
db.delete(filter={})

# add the loaded document chunks
db.add_documents(text_chunks)

### Verify product embeddings in SAP HANA Cloud

In [ ]:
# Query the table to verify embeddings
cursor = connection.cursor()
sql = f'SELECT VEC_TEXT, TO_NVARCHAR(VEC_VECTOR) FROM "{db.table_name}"'

cursor.execute(sql)
vectors = cursor.fetchall()

for vector in vectors:
    print(vector)

## Enhancing Query Responses

### Define a prompt template to provide context to queries

Define a prompt template to provide context to our prompts. Thus, when passed to the model, the template will add the necessary context to the prompt so that more accurate results are generated.

The answer should contain the requested information about products and their descriptions, formatted according to the specified JSON structure for further use in the SAP HANA JSON Document store.

> The created template for the prompt contains two variables - **context** and **question**. These variables will be replaced with the context and question in the upcoming steps.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

prompt_template = """use the following pieces of context to answer the question at the end. If you don't know the answer,
    just say you don't know, don't try to make up an answer. Format the results in a list of JSON items with the following keys:

        "PRODUCT_ID", 
        "PRODUCT_NAME",
        "CATEGORY",
        "DESCRIPTION",
        "UNIT_PRICE",
        "UNIT_MEASURE",
        "SUPPLIER_ID",
        "SUPPLIER_NAME",
        "LEAD_TIME_DAYS",
        "MIN_ORDER",
        "CURRENCY",
        "SUPPLIER_COUNTRY",
        "SUPPLIER_ADDRESS",
        "SUPPLIER_CITY",
        "CITY_LAT",
        "CITY_LONG",
        "RATING"
      
    
    The 'RATING' key value is an integer datatype ranging from 0 stars to 5 stars. Where 0 stars is 'bad' and 5 stars is 'excellent'. Do not include json markdown codeblock syntax in the results for example: ```json ```

    {context}

    question: {question}

    """


PROMPT = PromptTemplate(template = prompt_template, 
                        input_variables=["context", "question"]
                       )
    
chain_type_kwargs = {"prompt": PROMPT}

### Create the Conversational Retrieval Chain with SAP HANA Cloud vector engine


This code snippet integrates various components from the `langchain` library to create a retrieval-based question-answering (QA) system. Here's a breakdown of the key parts and their functionality:

- **Retriever Initialization:** The `db.as_retriever` function is used to initialize a retriever object with specific search arguments (`'k':20`), which likely defines the number of search results to consider.

- **Prompt Template :** The `PromptTemplate` was defined in the previous step that instructs how to use the context to answer a question. It emphasizes not to fabricate answers if the information is unavailable. The template also outlines the structure for the expected JSON output with various product and supplier details.


In [ ]:
question = "Find products with a rating of 4 and more."
retriever = db.as_retriever(search_kwargs={'k':20})

qa = RetrievalQA.from_chain_type(llm=llm,
                 retriever=retriever, 
                 chain_type="stuff",
                 chain_type_kwargs= chain_type_kwargs)

answer = qa.run(question)
print(answer)

## SAP HANA Cloud multi-modeling with Spatial and Vector engines

#### RAG results to SAP HANA Cloud Document Store (JSON)

Create a document collection **GX_RAG_PRODUCTS_DEV** to store the RAG results from the previous step. This script iterates over the list of product dictionaries, converting each one into a JSON string before inserting it into the collection. It then queries the collection to fetch and print all the inserted documents, allowing you to verify the insertion.

In [ ]:
# Sample dataset of product details
import json

jdata = json.loads(answer)
products = jdata
collection_name = "GX_RAG_PRODUCTS_DEV"

# Create a cursor
cursor = connection.cursor()

# Create a collection (document store)
try:
    cursor.execute(f"CREATE COLLECTION {collection_name}")
    print(f"{collection_name} Created!")
    
    # Insert JSON data into the collection
    for product in products:
        json_str = json.dumps(product)
        cursor.execute(f"INSERT INTO {collection_name} VALUES ('{json_str}')")

except :
    print(f"{collection_name} Recreated!")
    cursor.execute(f"DROP COLLECTION {collection_name}")
    cursor.execute(f"CREATE COLLECTION {collection_name}")

    # Insert JSON data into the collection
    for product in products:
        json_str = json.dumps(product)
        cursor.execute(f"INSERT INTO {collection_name} VALUES ('{json_str}')")


##### Query SAP HANA Cloud JSON Collection

This code snippet is designed to query a collection named **GX_RAG_PRODUCTS_DEV** to retrieve supplier information, including their ID, city, and location represented as a geospatial point.


In [ ]:
collection_name = "GX_RAG_PRODUCTS_DEV"
sql = f"SELECT PRODUCT_ID, PRODUCT_NAME, RATING FROM {collection_name}"


cursor.execute(sql)
query_result = cursor.fetchall()

for result in query_result:
    print(result)

cursor.close()

In [ ]:
collection_name = "GX_RAG_PRODUCTS_DEV"
sql = f"SELECT SUPPLIER_ID, SUPPLIER_CITY,SUPPLIER_ADDRESS,NEW ST_POINT(TO_DOUBLE(CITY_LONG),TO_DOUBLE(CITY_LAT)) AS SUPPLIER_LOCATION FROM {collection_name}"


cursor.execute(sql)
suppliers = cursor.fetchall()

for supplier in suppliers:
    print(supplier)

#### SAP HANA Cloud Spatial engine

This code snippet is designed to create a new table called **GX_SUPPLIER_LOCATIONS**, specifically for storing supplier location information. Note that **SUPPLIER_LOCATION** is a geospatial point representing the supplier's location, created using the `ST_POINT()` function. 

This point is constructed from two columns - **CITY_LONG** and **CITY_LAT** - which are converted to double precision numbers (representing longitude and latitude, respectively).


In [ ]:
table_name = "GX_SUPPLIER_LOCATIONS_DEV"
sql = f"""CREATE TABLE {table_name} AS (
               SELECT SUPPLIER_ID,
               SUPPLIER_CITY,
               NEW ST_POINT(
               TO_DOUBLE(CITY_LONG),
               TO_DOUBLE(CITY_LAT)) AS SUPPLIER_LOCATION,
               SUPPLIER_ADDRESS,
               PRODUCT_NAME,
               RATING

               FROM {collection_name})"""

try:
    cursor.execute(sql)
    print(f"{table_name} Created!")

except :
    print(f"{table_name} Recreated!")
    cursor.execute(f"DROP TABLE {table_name}")
    cursor.execute(sql)


##### Query to new table to fetch the geospatial data. 

This code snippet is designed to query SAP HANA Cloud using SQL to retrieve information about suppliers, including their IDs, city names, and geographic locations (latitude and longitude). Here's a breakdown of the code's functionality:

The latitude and longitude are extracted using the `ST_Y()` and `ST_X()` functions on the **SUPPLIER_LOCATION** column, which is presumably stored as a geospatial data type in the database.


In [ ]:
# Query to fetch the geospatial data
table_name = "GX_SUPPLIER_LOCATIONS_DEV"
sql = f"""SELECT SUPPLIER_ID,
               SUPPLIER_CITY,
               SUPPLIER_LOCATION.ST_Y() as latitudes, 
               SUPPLIER_LOCATION.ST_X() as longitudes,
               SUPPLIER_ADDRESS,
               PRODUCT_NAME,
               RATING 
            FROM {table_name}"""

cursor.execute(sql)

# Fetch all the results
points = cursor.fetchall()


for point in points:
    print(point)

# Close the cursor and connection
cursor.close()

#### Visualize supplier warehouse locations on a map 

The provided code snippet uses the Folium library in Python to create an interactive map centered around Germany. It initializes a map object centered at the coordinates for Germany (latitude 51.1657, longitude 10.4515) with a zoom level of 6. Then, it iterates over a dataset named `data`, which is expected to contain points of interest. Each point in the dataset includes a supplier ID, city name, latitude, and longitude.

For each point, the code creates a popup text that includes the city name and supplier ID. It then creates a marker at the point's latitude and longitude, attaches the popup text to this marker, and adds the marker to the map. The result is an interactive map where each marker represents a point of interest, and clicking on a marker displays a popup with additional information about that point.

In [ ]:
import folium

data = points

# Create a map centered around the average location
average_lat = sum([item[2] for item in data]) / len(data)
average_lon = sum([item[3] for item in data]) / len(data)
map = folium.Map(location=[average_lat, average_lon], zoom_start=7)

# Loop through each item in data to create markers
for supplier_id, city, lat, lon, address, product, rating in data:
    popup_text = f"Supplier ID: {supplier_id}<br>"
    popup_text += f"City: {city}<br>"
    popup_text += f"Address: {address}<br>"
    popup_text += f"Product: {product}<br>"
    popup_text += f"Rating: {rating}"
    
    # Add a marker to the map
    folium.Marker(
        [lat, lon],
        popup=folium.Popup(popup_text, max_width=300,min_width=300)
    ).add_to(map)

# Display the map
map
